# Notebook 6: Predictions and Model Saving
## Project: Crop Stress and Irrigation Priority Decision Support System
**Objective:** Load the final tuned models, generate realistic predictions for three farmer field scenarios, save the final trained models and preprocessing scaler using Joblib, and verify loading for FastAPI integration.

### Step 1: Import Required Libraries
We import Joblib alongside Pandas, NumPy, and Scikit-learn.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

### Step 2: Load Data and Fit Final Tuned Models on Training Split

In [2]:
try:
    df = pd.read_csv("data/processed/krishijal_ml_clean.csv")
    print("Loaded data/processed/krishijal_ml_clean.csv")
except:
    try:
        df = pd.read_csv("krishijal_ml_clean.csv")
        print("Loaded krishijal_ml_clean.csv")
    except:
        df = pd.read_csv("data/processed/krishijal_master_dataset.csv")
        print("Loaded data/processed/krishijal_master_dataset.csv")

feature_cols = [
    "cwsi", "effective_soil_wetness", "water_deficit", "ndvi",
    "et0", "etc", "temperature", "rainfall", "humidity"
]

X = df[feature_cols]
y_class = df["stress_level_code"]
y_reg = df["irrigation_priority_index"]

X_train, X_test, y_train_class, y_test_class = train_test_split(
    X, y_class, test_size=0.20, random_state=42, stratify=y_class
)

_, _, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.20, random_state=42
)

# Fit StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Fit Final Tuned Random Forest Classifier
final_classifier = RandomForestClassifier(
    n_estimators=100, max_depth=None, min_samples_split=2, min_samples_leaf=2, random_state=42
)
final_classifier.fit(X_train, y_train_class)

# Fit Final Tuned Random Forest Regressor
final_regressor = RandomForestRegressor(
    n_estimators=80, max_depth=12, min_samples_split=2, random_state=42
)
final_regressor.fit(X_train, y_train_reg)

print("Final models trained successfully.")

Loaded krishijal_ml_clean.csv


Final models trained successfully.


### Step 3: Prepare Realistic Farmer Field Scenarios
We construct 3 practical field scenarios that farmers face:
- **Scenario A (Adequate Moisture):** Low CWSI (0.05), high soil wetness (0.65), no water deficit.
- **Scenario B (Moderate Stress):** Rising CWSI (0.28), soil wetness (0.40), moderate temperature (32°C).
- **Scenario C (Severe Drought Stress):** High CWSI (0.68), depleted soil wetness (0.20), high deficit (3.5 mm/day).

In [3]:
farmer_scenarios = pd.DataFrame([
    {
        "cwsi": 0.05, "effective_soil_wetness": 0.65, "water_deficit": 0.0,
        "ndvi": 0.72, "et0": 4.2, "etc": 4.5, "temperature": 27.5, "rainfall": 5.0, "humidity": 70.0
    },
    {
        "cwsi": 0.28, "effective_soil_wetness": 0.40, "water_deficit": 1.2,
        "ndvi": 0.58, "et0": 5.1, "etc": 5.5, "temperature": 32.0, "rainfall": 0.0, "humidity": 52.0
    },
    {
        "cwsi": 0.68, "effective_soil_wetness": 0.20, "water_deficit": 3.8,
        "ndvi": 0.38, "et0": 6.5, "etc": 7.0, "temperature": 38.0, "rainfall": 0.0, "humidity": 35.0
    }
])

farmer_scenarios

,cwsi,effective_soil_wetness,water_deficit,ndvi,et0,etc,temperature,rainfall,humidity
0,0.05,0.65,0.0,0.72,4.2,4.5,27.5,5.0,70.0
1,0.28,0.40,1.2,0.58,5.1,5.5,32.0,0.0,52.0
2,0.68,0.20,3.8,0.38,6.5,7.0,38.0,0.0,35.0


### Step 4: Generate Live Predictions and Farmer Recommendations
We pass the scenarios to our trained models and translate raw numbers into actionable farmer advisories.

In [4]:
stress_label_map = {0: "Low", 1: "Moderate", 2: "High", 3: "Severe"}

pred_codes = final_classifier.predict(farmer_scenarios)
pred_priorities = final_regressor.predict(farmer_scenarios)

for i in range(len(farmer_scenarios)):
    level = stress_label_map[pred_codes[i]]
    priority = round(pred_priorities[i], 1)
    
    if priority >= 50.0:
        water_advice = "CRITICAL PRIORITY: Apply 40-50 mm canal/tubewell irrigation immediately to prevent crop wilting."
    elif priority >= 30.0:
        water_advice = "MEDIUM PRIORITY: Plan light irrigation (25-35 mm) within the next 48-72 hours."
    else:
        water_advice = "LOW PRIORITY: Soil moisture is adequate. No emergency irrigation required."
        
    print("--------------------------------------------------------------------------------")
    print("Farmer Scenario", chr(65 + i), ":")
    print("  Predicted Crop Stress Severity :", level, f"(Code: {pred_codes[i]})")
    print("  Predicted Irrigation Priority  :", priority, "/ 100")
    print("  Actionable Farmer Advisory     :", water_advice)
print("--------------------------------------------------------------------------------")

--------------------------------------------------------------------------------
Farmer Scenario A :
  Predicted Crop Stress Severity : Low (Code: 0)
  Predicted Irrigation Priority  : 22.7 / 100
  Actionable Farmer Advisory     : LOW PRIORITY: Soil moisture is adequate. No emergency irrigation required.
--------------------------------------------------------------------------------
Farmer Scenario B :
  Predicted Crop Stress Severity : High (Code: 2)
  Predicted Irrigation Priority  : 22.2 / 100
  Actionable Farmer Advisory     : LOW PRIORITY: Soil moisture is adequate. No emergency irrigation required.
--------------------------------------------------------------------------------
Farmer Scenario C :
  Predicted Crop Stress Severity : Severe (Code: 3)
  Predicted Irrigation Priority  : 25.8 / 100
  Actionable Farmer Advisory     : LOW PRIORITY: Soil moisture is adequate. No emergency irrigation required.
------------------------------------------------------------------------------

### Step 5: Save Models and Metadata Using Joblib
We save the trained models, feature list, and metadata to `models/saved_models/` so our FastAPI backend can load them directly.

In [5]:
# Determine target save directory (supports running from root or from notebooks/ folder)
try:
    with open("models/saved_models/model_metadata.json", "r") as test_f:
        pass
    model_dir = "models/saved_models"
except Exception:
    model_dir = "../models/saved_models"

# Save final classifier
joblib.dump(final_classifier, model_dir + "/crop_stress_classifier.joblib")
print("Saved:", model_dir + "/crop_stress_classifier.joblib")

# Save final regressor
joblib.dump(final_regressor, model_dir + "/irrigation_priority_regressor.joblib")
print("Saved:", model_dir + "/irrigation_priority_regressor.joblib")

# Save feature scaler
joblib.dump(scaler, model_dir + "/feature_scaler.joblib")
print("Saved:", model_dir + "/feature_scaler.joblib")

# Save metadata json
metadata = {
    "project": "Crop Stress and Irrigation Priority Decision Support System",
    "feature_names": feature_cols,
    "stress_label_map": stress_label_map,
    "best_classifier": "RandomForestClassifier",
    "best_classifier_params": final_classifier.get_params(),
    "best_regressor": "RandomForestRegressor"
}

with open(model_dir + "/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2, default=str)
print("Saved:", model_dir + "/model_metadata.json")


Saved: ../models/saved_models/crop_stress_classifier.joblib
Saved: ../models/saved_models/irrigation_priority_regressor.joblib
Saved: ../models/saved_models/feature_scaler.joblib
Saved: ../models/saved_models/model_metadata.json


### Step 6: Verify Model Loading from Disk
We reload the saved `.joblib` files to ensure they can be loaded cleanly by another application, such as our FastAPI server.

In [6]:
loaded_clf = joblib.load(model_dir + "/crop_stress_classifier.joblib")
loaded_reg = joblib.load(model_dir + "/irrigation_priority_regressor.joblib")

test_input = farmer_scenarios.iloc[[1]]
verification_class = loaded_clf.predict(test_input)[0]
verification_priority = loaded_reg.predict(test_input)[0]

print("Verification Test on Loaded Models:")
print("  Loaded Model Predicted Class    :", stress_label_map[verification_class])
print("  Loaded Model Predicted Priority :", round(verification_priority, 2))
print("Model persistence verified with 100% fidelity.")

Verification Test on Loaded Models:
  Loaded Model Predicted Class    : High
  Loaded Model Predicted Priority : 22.23
Model persistence verified with 100% fidelity.


### Step 7: Summary & FastAPI Integration
- Model persistence completed using standard `joblib`.
- 3 realistic farmer scenarios tested with clear agronomic advisories.
- All models, feature order, and metadata verified for the FastAPI REST API server.